# DSA Drill - Top K Frequent Elements (LC 347 - Medium)

> Given an integer array `nums` and an integer `k`, return the `k` **most frequent elements** (answer can be in any order).

**Examples**
1. `nums = [1,1,1,2,2,3]`, `k = 2` -> `[1, 2]`
2. `nums = [1]`, `k = 1` -> `[1]`
3. `nums = [4,4,4,6,6,7,7,7,7,9]`, `k = 2` -> `?` (derive by hand)

**Constraints:** `1 <= n <= 10^5` · `k` is guaranteed valid.

**Protocol (20:00 cap):** clarify -> verbalize the ladder (counting -> bucket sort -> heap) -> code -> tests.

**The ladder (verbalize all three, then implement bucket sort):**
1. Count with a hash map: `O(n)` - trivially.
2. Sort by frequency: `O(n log n)` - fine but not the lesson.
3. **Bucket sort on frequency:** frequencies are bounded by `n`, so make `n+1` buckets where bucket i holds all elements appearing exactly i times; walk from the highest bucket down collecting until k. `O(n)` time, `O(n)` space.

**Landmines:**
1. Why can bucket sort beat comparison sorting here? (state the bound that makes it legal)
2. Heap alternative: maintain a min-heap of size k -> `O(n log k)`. When is that BETTER than bucket sort? (think k << distinct count, or streaming)
3. Ties: the problem says any order - but your fuzz must not depend on tie order.

One hint allowed (costs the badge).

## Intuition (plain): invert the array-index trick

A frequency map tells you *how often* each element appears. But to get the top-k you would have to sort the values... unless you notice the bound: **a frequency can only be an integer from 1 to n**. So make buckets indexed BY frequency: `buckets[3]` holds every element that appears exactly 3 times.

Walk from `buckets[n]` down to `buckets[1]`, collecting elements until you have k. You never compare two elements - you index by their count, which is why it is O(n).

**The family:** "when the sort key is a small bounded integer, bucket-sort it" - same trick as counting sort, reused here.

In [ ]:
from collections import Counter

def top_k_frequent(nums: list[int], k: int) -> list[int]:
    """Return the k most frequent elements, O(n) via bucket sort on frequency."""
    counts = Counter(nums)
    n = len(nums)
    buckets = [[] for _ in range(n + 1)]      # freq -> elements with that freq
    for elem, freq in counts.items():
        buckets[freq].append(elem)
    out: list[int] = []
    for freq in range(n, 0, -1):              # highest frequency first
        for elem in buckets[freq]:
            out.append(elem)
            if len(out) == k:
                return out
    return out


def top_k_frequent_heap(nums: list[int], k: int) -> list[int]:
    """O(n log k) heap alternative - the answer to landmine 2."""
    import heapq
    counts = Counter(nums)
    heap = [(-freq, elem) for elem, freq in counts.items()]
    heapq.heapify(heap)
    return [heapq.heappop(heap)[1] for _ in range(min(k, len(heap)))]

In [ ]:
from collections import Counter

def _topk_freq_multiset(nums, k):
    """The multiset of frequencies any VALID top-k answer must have (order-free)."""
    freqs = sorted(Counter(nums).values(), reverse=True)
    return sorted(freqs[:k])

def _valid_topk(nums, k, result):
    """LC 347 allows ANY order; ties at the k-boundary make the element SET itself
    non-unique (two legal answers). So the reference checks the frequency multiset
    plus element validity, not element-set equality."""
    if len(result) != k or len(set(result)) != k:
        return False
    c = Counter(nums)
    return sorted(c[e] for e in result) == _topk_freq_multiset(nums, k)

assert top_k_frequent([1,1,1,2,2,3], 2) in ({1, 2}, [1, 2])
assert top_k_frequent([1], 1) == [1]

import random
random.seed(31)
for _ in range(300):
    n = random.randint(1, 60)
    nums = [random.randint(0, 9) for _ in range(n)]   # small alphabet -> boundary ties -> good fuzz
    k = random.randint(1, len(set(nums)))
    assert _valid_topk(nums, k, top_k_frequent(nums, k)), (nums, k, top_k_frequent(nums, k))
    assert _valid_topk(nums, k, top_k_frequent_heap(nums, k)), (nums, k, top_k_frequent_heap(nums, k))
print("fuzz 300/300: bucket sort and heap both valid (frequency-multiset check)")

In [ ]:
# Edge cases - run me, say WHY out loud
assert top_k_frequent([1, 2, 3, 4], 4) == [1, 2, 3, 4]  # all freq 1 -> k == n (bucket 1 only)
assert top_k_frequent([5, 5, 5, 5], 1) == [5]           # one element, max freq
assert set(top_k_frequent([0, 0, 1, 1], 2)) == {0, 1}   # tie on freq - any order is legal
assert top_k_frequent([7], 1) == [7]
print("edge cases pass OK")

## Post-mortem (fill in your own words)
- Brute force idea + complexity:
- Why bucket sort is legal here (the bound):
- Heap vs bucket: which to choose when, complexity of each:
- Tie-handling in the fuzz (why `set()` comparison):
- One-line pattern (copy into dsa/patterns.md):